# Named Entity Recognition for H3 (author_creator and caption structural tags)

Automatically find all person names in author_creator and caption fields and save them with their source file and volume information.

I have decided to work with spacy´s en_core_web_trf model.

For one, this model uses a real language model underneath (RoBERTa), additionally, while sm and lg use statistical features and word vectors, trf runs a full transformer that reads the whole sentence at once so it understands context.

Furthermore, en_core_web_trf has the highest NER accuracy of all spaCy English models and it is better on rare and uncommon names which may appear in caption and contributor fields. The transformer generalises better to uncommon names by reading surrounding context.

Additionally, because some caption fields contain fragmented or non-standard text, the choice was to use en_core_web_trf because Transformers are more robust to unusual token sequences than the statistical models.

However, one trade-off one needs to be aware of is that this model is significantly slower, however, in this case, this will only take several minutes and not hours on end.

For this Notebook, I have used the following documentation to help me write the code: 
- [Named Entity Recognition](https://spacy.io/usage/linguistic-features#named-entities)
- [English Model Comparison](https://spacy.io/models/en)
- [Counter for Counting Name Frequencies](https://docs.python.org/3/library/collections.html#collections.Counter)
- [Pandas for downloading and reshaping data](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.apply.html)
- [tqdm for the progress bar](https://tqdm.github.io/)

#### 1. Imports

I decided for spacy [because](https://www.geeksforgeeks.org/python/python-named-entity-recognition-ner-using-spacy/):

In [1]:
import pandas as pd
import spacy
import spacy_curated_transformers  # I need to load en_core_web_trf
import re
from collections import Counter
from tqdm import tqdm

# this lets pandas show a progress bar when I use .apply()
tqdm.pandas()

# load the transformer model — more accurate than the small model
# run once in terminal if not installed:
# pip install spacy-transformers
# python -m spacy download en_core_web_trf
nlp = spacy.load("en_core_web_trf")

#### 2. Loading the H3 Corpus

In [5]:
df = pd.read_csv("/Users/sophiehamann/master-thesis-code/data/processed/04_h3_clean.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nRegion types:")
print(df["region_type"].value_counts())
print()
print(df.head())

Shape: (2748, 4)

Columns: ['source_file', 'page_number', 'region_type', 'text']

Region types:
region_type
caption           1805
author_creator     943
Name: count, dtype: int64

                                         source_file  page_number  \
0  /Users/sophiehamann/master-thesis-code/data/ra...          NaN   
1  /Users/sophiehamann/master-thesis-code/data/ra...          NaN   
2  /Users/sophiehamann/master-thesis-code/data/ra...          NaN   
3  /Users/sophiehamann/master-thesis-code/data/ra...          NaN   
4  /Users/sophiehamann/master-thesis-code/data/ra...          NaN   

      region_type                                               text  
0  author_creator                                               Joan  
1  author_creator                                  Howardena Pindell  
2         caption  Howardena Pindell. Yes — No. Pen and ink on ac...  
3         caption  Gloria Rayl. Untitled. Clay masks pulled from ...  
4  author_creator                                

#### 3. Adding Issue Number and Volume Label

In [6]:
def get_issue_number(filepath):
    """Extract the issue number from the filename, e.g. heresies_08_combined → 8"""
    match = re.search(r"heresies_(\d+)_combined", filepath)
    return int(match.group(1)) if match else None

def get_volume(issue_nr):
    """Map issue number to a volume label"""
    if issue_nr is None:
        return None
    if issue_nr <= 4:  return "Vol1_1977-1978"
    if issue_nr <= 8:  return "Vol2_1978-1979"
    if issue_nr <= 12: return "Vol3_1980-1981"
    if issue_nr <= 16: return "Vol4_1981-1983"
    if issue_nr <= 20: return "Vol5_1984-1985"
    if issue_nr <= 24: return "Vol6_1987-1989"
    if issue_nr <= 27: return "Vol7_1990-1993"
    return None

df["issue"]  = df["source_file"].apply(get_issue_number)
df["volume"] = df["issue"].apply(get_volume)

print(df["volume"].value_counts())

volume
Vol1_1977-1978    531
Vol3_1980-1981    411
Vol4_1981-1983    403
Vol2_1978-1979    382
Vol6_1987-1989    367
Vol5_1984-1985    359
Vol7_1990-1993    295
Name: count, dtype: int64


#### 4. Extracting person names with Spacy (author_creator and caption separately)

In [7]:
def extract_persons(text):
    if pd.isna(text):
        return []
    doc = nlp(text)
    return [ent.text for ent in doc.ents if ent.label_ == "PERSON"]

# split into two dataframes
df_authors  = df[df["region_type"] == "author_creator"].copy()
df_captions = df[df["region_type"] == "caption"].copy()

print(f"Author/creator fields: {len(df_authors)} rows")
print(f"Caption fields: {len(df_captions)} rows")

# run NER on each separately
df_authors["persons"]  = df_authors["text"].progress_apply(extract_persons)
df_captions["persons"] = df_captions["text"].progress_apply(extract_persons)

Author/creator fields: 943 rows
Caption fields: 1805 rows


100%|██████████| 1805/1805 [00:45<00:00, 39.61it/s]


#### 5. Basic Cleaning and Removing Noise (different rules for author_creator and caption)

In [8]:
def clean_authors(persons):
    cleaned = []
    for name in persons:
        name = name.replace("'s", "").strip()
        if any(char.isdigit() for char in name):
            continue
        # author_creator fields: require 2+ words (hyphens count)
        if len(name.replace("-", " ").split()) < 2:
            continue
        cleaned.append(name)
    return cleaned

def clean_captions(persons):
    cleaned = []
    for name in persons:
        name = name.replace("'s", "").strip()
        if any(char.isdigit() for char in name):
            continue
        # same 2-word rule as author fields — hyphens count as word separators
        if len(name.replace("-", " ").split()) < 2:
            continue
        cleaned.append(name)
    return cleaned

df_authors["persons"]  = df_authors["persons"].apply(clean_authors)
df_captions["persons"] = df_captions["persons"].apply(clean_captions)

#### 6. Checking results for both

In [9]:
author_persons  = [name for persons in df_authors["persons"]  for name in persons]
caption_persons = [name for persons in df_captions["persons"] for name in persons]

print("=== AUTHOR/CREATOR FIELDS ===")
print(f"Total mentions: {len(author_persons)}, Unique: {len(set(author_persons))}")
for name, count in Counter(author_persons).most_common(20):
    print(f"  {name}: {count}")

print("\n=== CAPTION FIELDS ===")
print(f"Total mentions: {len(caption_persons)}, Unique: {len(set(caption_persons))}")
for name, count in Counter(caption_persons).most_common(20):
    print(f"  {name}: {count}")

=== AUTHOR/CREATOR FIELDS ===
Total mentions: 851, Unique: 775
  JOAN BRADERMAN: 11
  Lucy R. Lippard: 6
  Valerie Harris: 4
  Su Friedrich: 4
  hattie gossett: 4
  Barbara Ehrenreich: 3
  Harmony Hammond: 3
  Adrienne Rich: 3
  Amy Sillman: 3
  Olga Broumas: 3
  AISHA ESHE: 3
  Lorraine Schein: 3
  Michele Russell: 2
  Lyn Hughes: 2
  Elizabeth Zelvin: 2
  May Stevens: 2
  Karen Brodine: 2
  Lizzie Borden: 2
  BERTA NAVARRO: 2
  NORA DE IZQUE: 2

=== CAPTION FIELDS ===
Total mentions: 1509, Unique: 1162
  Su Friedrich: 8
  Mary Beth Edelson: 8
  Ana Mendieta: 7
  eeva-inkeri: 6
  Carolien Stikker: 6
  V. E. Browne: 5
  Lyn Hughes: 5
  Margaret Randall: 5
  Nancy Spero: 5
  Roberta Neiman: 5
  Clythia Fuller: 5
  Sondra Segal: 5
  Michele Godwin: 5
  Betye Saar: 4
  Nicky Lindeman: 4
  Diana Agosta: 4
  Susan Kleckner: 4
  Cecilia Vicuña: 4
  Lilly Reich: 4
  Ann Wilson: 4


#### 7. Fix false positives and then combine both

In [ ]:
# I filled the false positives by looking at the most common names in both lists and checking which ones are not actually people.
# I also added some name variations to the mapping.
#false_positives = {
    #"Patron Saint",
    # add more after inspecting step 6 output
#}

name_mapping = {
    "Lucy R. Lippard": "Lucy Lippard",
    # add more after inspecting step 6 output
}

def clean_name(name):
    # strip leading/trailing punctuation like hyphens, commas, quotes
    name = re.sub(r'^[^a-zA-ZÀ-ÿ]+', '', name)  # remove non-letters from start
    name = re.sub(r'[^a-zA-ZÀ-ÿ]+$', '', name)  # remove non-letters from end
    name = name.strip()

    if not name:
        return None
    if name in false_positives:
        return None
    return name_mapping.get(name, name)

for df_ in [df_authors, df_captions]:
    df_["persons"] = df_["persons"].apply(
        lambda names: [n for n in (clean_name(n) for n in names) if n is not None]
    )

# combine back into one dataframe
df_combined = pd.concat([df_authors, df_captions])

#### 8. One row per person mention

In [14]:
rows = []
for _, row in df_combined.iterrows():
    for person in row["persons"]:
        rows.append({
            "source_file": row["source_file"],
            "issue":       row["issue"],
            "volume":      row["volume"],
            "region_type": row["region_type"],
            "person":      person,
        })

df_persons = pd.DataFrame(rows)
print("Shape:", df_persons.shape)
print(df_persons.head(10))

Shape: (2360, 5)
                                         source_file  issue          volume  \
0  /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   
1  /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   
2  /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   
3  /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   
4  /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   
5  /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   
6  /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   
7  /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   
8  /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   
9  /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   

      region_type                  person  
0  author_creator       Howardena Pindell  
1  author_creator       M

#### 9. Saving

In [15]:
out_path = "/Users/sophiehamann/master-thesis-code/data/processed/09_h3_contributors.csv"
df_persons.to_csv(out_path, index=False)
print("Saved to:", out_path)

Saved to: /Users/sophiehamann/master-thesis-code/data/processed/09_h3_contributors.csv
